# KosenMap Website 取扱説明書 —— 目次

**この Website(公開ページ・管理画面・本番ホスト)を動かすための取扱説明書。**
読むだけでなく、**書いてあるコマンドをそのセルから実行できる。**

対象は本番 `ito4.jp`(`/opt/kosenmap`)と、この PC の `server/scripts`。
**繋ぐ利用者は 2 つ**(2026-09-18 に分けた。[12](12-hardening-2026-09-15.ipynb) §7-4 B): 配備・`%%host`・控えは **`kmops`**(鍵 `~\.ssh\km_ops`。docker あり・**sudo なし**)、`sudo` が要るセルは **`km`**(鍵 `~\.ssh\km_vps`。**docker なし**)。
Android アプリは別リポジトリなので、ここには入っていない。

| ノートブック | 何が書いてあるか |
|---|---|
| [00-start](00-start.ipynb) | **この取扱説明書の使い方。** 準備と、セルの型  |
| [01-daily-check](01-daily-check.ipynb) | **日々の確認。** 自己検査・ホストの様子・メールが届いているか |
| [02-deploy](02-deploy.ipynb) | **配備。** 下見・配備・後片付け・困ったとき |
| [03-backup](03-backup.ipynb) | **バックアップと復元。** 取る・開く・週次タスク・添付の復号・戻す |
| [04-host-jobs-and-mail](04-host-jobs-and-mail.ipynb) | **ホストの定期処理とメール。** cron の時刻表・届くメールの一覧・試しに送る |
| [05-containers](05-containers.ipynb) | **コンテナの更新と追加。** Logto の上げ方・固定・足し方 |
| [06-emergency](06-emergency.ipynb) | **もしものとき。** まず叩く1本と、症状別の見どころ |
| [07-map-qr](07-map-qr.ipynb) | **地図の配信と QR。** |
| [08-architecture](08-architecture.ipynb) | **どう出来ているか。** 構造・設定の読み方・落とし穴・設計の約束 |
| [09-new-host](09-new-host.ipynb) | **新しいホストを作る・移す。** VPS の構築から切り替えまで |
| [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb) | **診断の報告書(2026-09-14)。** 何を見つけて何を直したか・本番で実行する手順 |
| [11-getting-started](11-getting-started.ipynb) | **新しく使う人の入口。** どこから読み、最初に何をするか |
| [12-hardening-2026-09-15](12-hardening-2026-09-15.ipynb) | **多層防御の底上げ(2026-09-15)。** 網の分割・read_only・Soketi の Node 24・Logto の DB 利用者・管理画面の別オリジン・本番への当て方 |
| [13-local-env](13-local-env.ipynb) | **ローカル環境(LAN の検証機)。** `.env` の `KM_ENV=local` で、Let's Encrypt を使わずに本番と同じ構成を立てる |
| [14-domain-ito4](14-domain-ito4.ipynb) | **ito4.jp に統一する(2026-09-17)。** 旧 ito8795.com は同時に手放す・audience も https://ito4.jp/api へ・管理画面は admin.ito4.jp・外部スキャンの指摘 |
| [15-staff-org](15-staff-org.ipynb) | **教職員の自動付与(2026-09-18〜)。** 学校ドメインの JIT で Logto の組織へ入れ、組織ロールで「地図の錠を通る・教職員氏名を見る・閲覧不可の地点を見る」を与える |

記録として残している文書:

| 文書 | 中身 |
|---|---|
| [status.md](status.md) | **いまどうなっているか。** 実測値・踏んだ罠・残作業 |
| [plan.md](plan.md) | **これから何をするか。** 優先順と、やらないと決めたこと |
| [../Old/docs/](../Old/docs/) | 以前の手順書(md)。**細部の経緯はここ**。ノートブックに移した内容の元 |

# ホストの定期処理とメール

**機械は調べて知らせるだけ。当てるのは人。** 何がいつ動き、どんなメールが来るかの一覧。

> 使い方は [00-start.ipynb](00-start.ipynb)。**まず下のセルを1回実行する。**

| 印 | 意味 |
|---|---|
| 🟢 | **読むだけ。** 何も変えない。迷ったらここから |
| 🟡 | **手元が変わる。** この PC にファイルを作る・登録する。本番には触れない |
| 🔴 | **本番が変わる。** 実行前に `yes` の入力を求める |
| 🔑 | **別の窓で開く。** sudo のパスワードなど対話が要るもの |

In [ ]:
# 最初に1回だけ実行する(%%ps / %%host / %%terminal が使えるようになる)
import sys, pathlib
for _d in (pathlib.Path.cwd(), pathlib.Path.cwd() / 'docs', pathlib.Path.cwd() / 'server' / 'docs'):
    if (_d / 'km_nb.py').exists():
        sys.path.insert(0, str(_d))
        break
import km_nb
km_nb.load()

## 1. 時刻表(`/etc/cron.d/kosenmap-updates`、版 6)

| 時刻 | 何を | メール |
|---|---|---|
| **月〜土 2:40** | バックアップ(`host-backup.sh --notify`) | **失敗したときだけ** |
| **日曜 2:40** | バックアップ + 実行記録の添付(`--heartbeat --attach-logs`) | **必ず**(「取れています」を毎週届ける) |
| **毎日 3:47** | 証明書の更新(`send-log.sh --only-failure --run "host-cert.sh renew"`)。版 6 で追加 | **失敗したときだけ**(残り 30 日を切るまでは「更新の時期ではない」で終わる) |
| **毎月1日 4:07** | 証明書の状態(`send-log.sh --run "host-cert.sh status"`)。版 6 で追加 | **必ず**(残り日数と、nginx が出している証明書が一致しているか) |
| 毎日 4:17 | コンテナの更新を調べる(`check-updates.sh --notify`) | 変化があったとき(同じ知らせは7日おき) |
| **毎月1日 4:23** | 同上(`--heartbeat`) | **必ず** |
| 毎日 8:23 | ホストのセキュリティ(`host-security-check.sh --notify`) | 問題があったとき(同じ知らせは7日おき) |
| **毎月1日 8:29** | 同上(`--heartbeat`) | **必ず** |

**`--heartbeat` は、問題が無くても送る。** 沈黙を「正常」と読ませないため ——
仕掛けが止まっていても、壊れた側からは何も来ない。**届くはずの便りが来ない週・月に気づける。**

**証明書を cron でも回すのは、certbot コンテナのループが失敗しても黙って眠り続け、更新しても nginx は reload まで古い証明書を出し続けるため。**
HSTS を有効にしてあるので、**切れた時点で誰もサイトに入れなくなる。** 更新が済むと `host-cert.sh` が nginx を reload し、出している証明書の指紋まで突き合わせる。手で見るなら §4 の「証明書」。

実行記録は `/var/log/kosenmap/`(root:root 750。`backup.log` / `updates.log` / `security.log` / 版 6 で足した `cert.log`)。同じ知らせを繰り返さないための状態は `/var/lib/kosenmap/`。

## 2. 届くメールの一覧

| 件名(例) | いつ | 出すもの |
|---|---|---|
| `[KosenMap] バックアップを取りました (636 KB)・実行記録を添付` | 日曜。問題があれば「気になる点があります」 | `host-backup.sh` → `notify-backup.php` |
| `[KosenMap] バックアップに失敗しました` | 月〜土、失敗したとき | 同上 |
| `[KosenMap] 新しい版があります` / `修正版が出ています` / `ビルド元が更新されています` | 変化があったとき・毎月1日 | `check-updates.sh` → `notify-update.php` |
| `[KosenMap] ホストの再起動が必要です` など / `ホストのセキュリティは問題ありません(定期のお知らせ)` | 問題があったとき・毎月1日 | `host-security-check.sh` → `notify-security.php` |
| `[KosenMap] 証明書の更新: 失敗 (ホスト名)` | 毎日 3:47、**失敗したときだけ**(証明書の更新) | `host-cert.sh renew` → `send-log.sh` → `notify-log.php` |
| `[KosenMap] 証明書の状態: 成功 (ホスト名)`(残りが 14 日を切った・nginx が古い証明書を出しているときは「失敗」) | 毎月1日 4:07(証明書の月次。**必ず**届く) | `host-cert.sh status` → 同上 |
| `[KosenMap] 手元の週次バックアップ (ITO-PC): 失敗` | この PC の週次が失敗したとき | `backup-task-run.ps1` → `send-log.sh` → `notify-log.php` |
| `[KosenMap] お問い合わせ: <件名>` | 公開ページのフォームから | `contact.php` |
| Logto の確認コードなど | サインイン・登録のとき | **Logto 自身**(SMTP コネクター) |

ホストのセキュリティの知らせには、2026-09-14 から **fail2ban** と **Logto Console の2段階認証**の項目も載る(§4)。

**メールはすべて `docker compose exec web php …` で出す。** `.env` を docker compose が読めないと、全部出ない。

## 3. 仕込み(host-updates-setup.sh)

**root で走らせる。** 引数なしは調べるだけ、`--fix` で仕込む(cron の書き直し・持ち主と権限の修正・自動更新の設定)。
`deploy-to-host.ps1` で `host-updates-setup.sh` を変えたら、`--fix` をもう一度流すこと。

**版 6(2026-09-14)で証明書の2行を足した。** 版 5 のままのホストでは、調べるだけのセルが「定期実行が古い形です(版 6 ではありません)」と出す。
配備したあと、手元の `host-setup.ps1 -Fix` で実行ビットを付けてから `--fix` を流す(cron は `host-cert.sh` を `send-log.sh` 越しに呼ぶので、**2本とも実行ビットが要る**。`--fix` も付け直す)。

### 調べるだけ

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-updates-setup.sh --path /opt/kosenmap"

### 仕込む / 直す

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal --confirm "本番の cron と自動更新の設定を書き直します(sudo のパスワードを聞かれます)"
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-updates-setup.sh --path /opt/kosenmap --fix"

## 4. 手で調べる(送らない)

### コンテナの更新

種類の見方は [05-containers](05-containers.ipynb) の §4(`release` / `digest` / `series` / `base`)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/check-updates.sh --path /opt/kosenmap

### ホストのセキュリティ(`kmops` の権限で)

SSH の入口の検査は root のときだけ走る。

2026-09-14 に2つ足した(どちらも `kmops` の権限でも見られる):

| 項目 | ★ になるとき | 直し方 |
|---|---|---|
| **fail2ban** | 入っているのに動いていない(入っていないこと自体は数えない)。本番は起動直後に落ちたまま(failed)だった | `sudo fail2ban-client -t` で設定の誤りを直してから `sudo systemctl enable --now fail2ban` |
| **Logto Console のアカウント** | admin テナント(Console に入るアカウント)の MFA が `Mandatory` でない。サイトの利用者(default テナント)とは別の設定で、**Console の画面からは変えられない** | 先に控えを取り、認証アプリを手元に用意してから、postgres コンテナの psql で admin テナントの MFA を必須にする(SQL は届いたメールの本文と [10-security-review-2026-09-14](10-security-review-2026-09-14.ipynb)) |

同じ知らせは、状態が変わるまで繰り返さない(ほかの項目と同じく7日おきの念押しだけ)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 300
./scripts/host-security-check.sh --path /opt/kosenmap

### ホストのセキュリティ(root で全部)

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo /opt/kosenmap/scripts/host-security-check.sh --path /opt/kosenmap"

### 証明書(host-cert.sh)

**残り日数と、nginx が 443 番で実際に出している証明書**を突き合わせる。更新しても reload されなければ古い証明書が出続けるので、ファイルの期限だけでは足りない。

| 行の頭 | 意味 |
|---|---|
| `★` | 残り 14 日未満 / 443 番から受け取れない / **nginx が古い証明書を出している**(`renew` で直る)。cron なら「失敗」で届く |
| `注` | 残り 21 日未満。certbot は残り 30 日で更新するので、**更新が1週間以上効いていない** |
| `更新方式: standalone` | 素の `certbot renew` が失敗する設定。下の `fix-conf` で揃える |

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 120
./scripts/host-cert.sh --path /opt/kosenmap status

### 試しの更新(本物の証明書は変えない)

試験用の発行元(staging)に問い合わせ、80 番の `/.well-known/acme-challenge/` が届くかまで手順を通す。**発行回数の上限を消費しない。**
certbot コンテナのループと重なったら、60 秒待ってやり直す(最大 2 回)。試験用の発行元の登録と、確認用の一時ファイルが letsencrypt の置き場に出来るほかは何も変えない。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 420
./scripts/host-cert.sh --path /opt/kosenmap renew --dry-run

### 更新の設定を webroot に揃える(fix-conf)

本番の `renewal/ito8795.com.conf` は `authenticator = standalone` のままだった(2026-09-14 に実測)。
コンテナのループと cron は `--webroot` を渡すので更新は通るが、**素の `certbot renew` は 80 番を掴めずに失敗する** —— 手で叩いた人が「更新が壊れている」と誤解する。**証明書そのものは変わらない。**

まず下見(何を変えるかを出すだけ)。

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host --timeout 120
./scripts/host-cert.sh --path /opt/kosenmap fix-conf --dry-run

書き換える。`certbot reconfigure` が試験用の発行元で1回通してから、`renewal/*.conf` を書き換える。済んだら状態を出す。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "証明書の更新の設定(renewal/*.conf)を webroot に書き換えます(証明書そのものは変えません)" --timeout 300
./scripts/host-cert.sh --path /opt/kosenmap fix-conf </dev/null
./scripts/host-cert.sh --path /opt/kosenmap status </dev/null

### 実行記録を読む

🔑 **別の窓で開く** —— sudo のパスワードなど、対話が要るものです。

In [ ]:
%%terminal
ssh -t -i "$env:USERPROFILE\.ssh\km_vps" km@ito4.jp "sudo tail -n 40 /var/log/kosenmap/backup.log /var/log/kosenmap/updates.log /var/log/kosenmap/security.log /var/log/kosenmap/cert.log"

## 5. 試しに送る

**本物の宛先へ1通ずつ届く。** `kmops` の権限(`%%host`)で送ると、最後に `cannot create /var/lib/kosenmap/…: Permission denied` と出る ——
「送った」という状態を書けなかっただけで、cron(root)には影響しない。

### 更新のお知らせ

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "更新のお知らせを1通送ります" --timeout 300
./scripts/check-updates.sh --path /opt/kosenmap --notify --heartbeat 2>&1 | tail -12

### セキュリティのお知らせ

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "セキュリティのお知らせを1通送ります" --timeout 300
./scripts/host-security-check.sh --path /opt/kosenmap --notify --heartbeat 2>&1 | tail -6

### ログのメール(send-log.sh)

**何のログでも受ける口。** 秘密らしき値は `log-notice.php` が伏せてから送るが、**伏せ字は完璧ではない**(知っている形しか消せない)。
終了コードごと送りたいときは `--run "<コマンド>"` に任せる(sh にはパイプの左の終了コードを取る手段が無い)。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "試しのログのメールを1通送ります"
printf '試しの記録です\n文字化けの確認: 指紋を業者のコンソールと見比べる\n' \
  | ./scripts/send-log.sh --path /opt/kosenmap --label '【試験】ログの送信' --status ok

### 問い合わせと同じ送り方

フォームは reCAPTCHA があるので、同じ送信の関数を直接呼ぶ。受信箱(DB)には入らない。

🔴 **本番が変わる** —— 実行前に `yes` の入力を求めます。

In [ ]:
%%host --confirm "問い合わせと同じ形の試験メールを1通送ります"
docker compose exec -T web php <<'PHP'
<?php
require '/var/www/html/lib/mailer.php';
km_mail_send(
    km_mail_admin_to(),
    '[KosenMap] お問い合わせ: 【試験】送信経路の確認',
    "(これは試験です。フォームと同じ送信の関数を直接呼んでいます。受信箱には入っていません)\n\n文字化けの確認: 指紋を業者のコンソールと見比べる"
);
echo "送りました\n";
PHP

### 届いたか(送信サーバーの記録)

🟢 **読むだけ** —— 何も変えません。

In [ ]:
%%host
docker logs -t --since 1h km-mailserver 2>&1 \
  | grep -E 'status=(sent|bounced|deferred)' \
  | sed -E 's/^([0-9-]+)T([0-9:]{8})[^ ]* .*to=<[^@>]*@([^>]*)>.*status=([a-z]+).*/\1 \2 UTC  …@\3  \4/'

## 6. 届かないとき

| 見るもの | 打つ手 |
|---|---|
| `.env` を docker compose が読めるか | [00-start](00-start.ipynb) の「ホストへ繋がるか」。読めなければ値の引用符・記号を外す |
| cron が動いているか | [01-daily-check](01-daily-check.ipynb) の「定期処理の仕込み」。版 6・root:root 644 |
| 送信サーバーが `status=sent` を出しているか | 上のセル。`deferred` / `bounced` なら相手側で拒まれている |
| 証明書の知らせが来ない | 更新の知らせは**失敗したときだけ**。毎月1日の「証明書の状態」が届いていれば仕掛けは動いている。§4 の「証明書」で手で見る |
| 迷惑メールに入っていないか | DKIM・SPF・DMARC はすべて pass(2026-09-13 に確認) |

送信サーバーの設計・DNS・DKIM の登録は [../Old/docs/mail-server.md](../Old/docs/mail-server.md)、
Logto のメール(SMTP コネクターとテンプレート9種)は [../Old/docs/logto-smtp.md](../Old/docs/logto-smtp.md)。